In [4]:
(82.9-75.7)/100*44

3.168000000000001

In [2]:
# 读取jsonl文件
import json

def read_first_n_lines_jsonl(file_path, n):
    """
    读取并解析一个 JSONL 文件的前 n 行。

    参数:
    file_path (str): JSONL 文件的路径。
    n (int): 需要读取的行数。

    返回:
    list: 一个包含已解析的JSON对象（字典）的列表。
    """
    data = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                # 如果已经读取了 n 行，则停止循环
                if i >= n:
                    break
                # 解析当前行并添加到列表中
                data.append(json.loads(line.strip()))
    except FileNotFoundError:
        print(f"错误：文件 '{file_path}' 未找到。")
    except json.JSONDecodeError as e:
        # 如果某一行不是有效的JSON，会捕获错误
        print(f"错误：解析第 {i+1} 行时发生JSON解码错误: {e}")
    except Exception as e:
        print(f"发生未知错误: {e}")
        
    return data




# 2. 设置文件路径和要读取的行数
file_path = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_responses_llm.jsonl'
num_lines_to_read = 5

# 3. 调用函数读取文件
first_lines = read_first_n_lines_jsonl(file_path, num_lines_to_read)

# 4. 打印结果
if first_lines:
    print(f"成功读取文件 '{file_path}' 的前 {len(first_lines)} 行内容：")
    for item in first_lines:
        print(item)

成功读取文件 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_responses_llm.jsonl' 的前 5 行内容：
{'average_llm_score': 0.7570202110534788}
{'id': 1, 'image_id': '0e750975c41bf199c11c1ea035ecb5cf', 'task_type': 'single-turn-vqa', 'question': 'How does The Fun Experts handle my email privacy?', 'ground_truth': "The Fun Experts assure users of their commitment to email privacy by explicitly stating on their webpage that they never share your data with any third parties. This promise is highlighted in a section of the site that likely aims to reassure visitors when they are considering signing up for the Fun Experts Newsletter. The company's dedication to protecting customer data is a key part of their privacy practices, especially in relation to the collection and use of email addresses. This information is typically presented in close proximity to where users can input their email addresses to subscribe to the newsletter, ensuring that the privacy assurance is see

In [10]:
import json

def save_filtered_data(file1_path, file2_path, output_path):
    """
    比较两个jsonl文件，当满足条件（file1评分为0，file2评分为1）时，
    提取'question'和'ground_truth'字段，并保存到新的jsonl文件中。

    Args:
        file1_path (str): 第一个jsonl文件的路径。
        file2_path (str): 第二个jsonl文件的路径。
        output_path (str): 用于保存结果的新jsonl文件的路径。

    Returns:
        int: 成功保存的数据条数。
    """
    saved_count = 0
    print(f"开始筛选数据，并将结果保存到 {output_path} ...")

    with open(file1_path, 'r', encoding='utf-8') as f1, \
         open(file2_path, 'r', encoding='utf-8') as f2, \
         open(output_path, 'w', encoding='utf-8') as f_out:
        
        next(f1, None)
        next(f2, None)

        for line_num, (line1, line2) in enumerate(zip(f1, f2), start=2):
            try:
                data1 = json.loads(line1)
                data2 = json.loads(line2)

                if data1.get('llm_score') == 0 and data2.get('llm_score') == 1:
                    # 1. 创建一个新的字典，只包含需要的字段。
                    #    数据从 data2 中提取，因为它代表了改进后的版本。
                    filtered_entry = {
                        "question": data2.get("question"),
                        "image_id": data2.get("image_id"),
                        "ground_truth": data2.get("ground_truth")
                    }

                    # 2. 将这个新字典转换为JSON格式的字符串。
                    #    ensure_ascii=False 确保中文字符能被正确写入，而不是转义码。
                    output_line = json.dumps(filtered_entry, ensure_ascii=False)

                    # 3. 将字符串写入文件，并手动添加一个换行符 \n
                    f_out.write(output_line + '\n')
                    saved_count += 1
            
            except json.JSONDecodeError:
                print(f"警告：第 {line_num} 行解析JSON失败，已跳过。")
    
    return saved_count

# --- 主执行程序 ---
# 1. 定义输入文件路径
file1_path = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_responses_llm.jsonl'
file2_path = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm.jsonl'

# 2. 定义一个更有描述性的输出文件名
output_file_path = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_wrong_72b_right.jsonl'

# 3. 执行函数
num_saved = save_filtered_data(file1_path, file2_path, output_file_path)

# 4. 打印最终结果
if num_saved > 0:
    print(f"\n操作完成！成功将 {num_saved} 条数据保存到文件 '{output_file_path}'。")
else:
    print("\n操作完成。没有找到符合条件的数据。")

开始筛选数据，并将结果保存到 /mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_wrong_72b_right.jsonl ...

操作完成！成功将 5249 条数据保存到文件 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_wrong_72b_right.jsonl'。


In [7]:
import json

def find_missing_data(gemini_file, other_file, output_file):
    """
    Finds data present in other_file but not in gemini_file based on the question.

    Args:
        gemini_file (str): Path to the gemini format jsonl file.
        other_file (str): Path to the other jsonl file to compare against.
        output_file (str): Path to the output jsonl file.
    """
    gemini_questions = set()
    gemini_data_count = 0
    with open(gemini_file, 'r', encoding='utf-8') as f:
        for line in f:
            gemini_data_count += 1
            try:
                data = json.loads(line)
                for item in data:
                    if item.get('role') == 'user':
                        for content in item.get('content', []):
                            if content.get('type') == 'text':
                                gemini_questions.add(content['text'])
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON line in {gemini_file}: {line.strip()}")
    
    print(f"Total number of data entries in {gemini_file}: {gemini_data_count}")

    missing_data = []
    other_data_count = 0
    with open(other_file, 'r', encoding='utf-8') as f_other, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        for line in f_other:
            other_data_count += 1
            try:
                data = json.loads(line)
                question = data.get('question')
                if question and question not in gemini_questions:
                    new_data = {
                        "question": question,
                        "image_id": data.get("image_id"),
                        "ground_truth": data.get("ground_truth")
                    }
                    missing_data.append(new_data)
                    f_out.write(json.dumps(new_data) + '\n')
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON line in {other_file}: {line.strip()}")
    
    print(f"Total number of data entries in {other_file}: {other_data_count}")
    print(f"Total number of missing data entries: {len(missing_data)}")
    print(f"Missing data has been saved to {output_file}")

# You would replace these file paths with your actual file paths
find_missing_data(
    "/mnt/petrelfs/sunhaoyu/visual-code/Tool-Factory-Filter/tool_server/tf_eval/scripts/logs/ckpt/web/gemini_test_format_conversation.jsonl",
    "/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box_ratings.jsonl",
    "/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/sft_to_rl.jsonl"
)

Total number of data entries in /mnt/petrelfs/sunhaoyu/visual-code/Tool-Factory-Filter/tool_server/tf_eval/scripts/logs/ckpt/web/gemini_test_format_conversation.jsonl: 1141
Total number of data entries in /mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered_box_ratings.jsonl: 1870
Total number of missing data entries: 713
Missing data has been saved to /mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/sft_to_rl.jsonl


In [9]:
import json
from collections import Counter

def count_unique_unseen_images(file1_path, file2_path):
    """
    计算文件2中，image_id没有在文件1的llm_score为0的数据中出现过，
    并且该image_id在文件2中只出现过一次的数据条数。

    Args:
        file1_path (str): 第一个jsonl文件的路径。
        file2_path (str): 第二个jsonl文件的路径。

    Returns:
        int: 符合所有条件的数据条数。
    """
    # --- 步骤 1: 读取文件1，收集所有llm_score为0的image_id ---
    zero_score_image_ids = set()
    try:
        with open(file1_path, 'r', encoding='utf-8') as f1:
            for line in f1:
                try:
                    data = json.loads(line)
                    if data.get("llm_score") == 0:
                        zero_score_image_ids.add(data.get("image_id"))
                except json.JSONDecodeError:
                    print(f"警告：文件1中有一行无法解析为JSON: {line.strip()}")
    except FileNotFoundError:
        print(f"错误：找不到文件 {file1_path}")
        return -1

    # --- 步骤 2: 读取文件2，统计其中所有image_id的出现频率 ---
    image_ids_in_file2 = []
    try:
        with open(file2_path, 'r', encoding='utf-8') as f2:
            for line in f2:
                try:
                    data = json.loads(line)
                    image_ids_in_file2.append(data.get("image_id"))
                except json.JSONDecodeError:
                    print(f"警告：文件2中有一行无法解析为JSON: {line.strip()}")
    except FileNotFoundError:
        print(f"错误：找不到文件 {file2_path}")
        return -1
    
    # 使用Counter快速统计每个image_id的出现次数
    image_id_counts = Counter(image_ids_in_file2)

    # --- 步骤 3: 筛选出同时满足两个条件的数据 ---
    count = 0
    # 遍历在文件2中只出现过一次的image_id
    for image_id, frequency in image_id_counts.items():
        if frequency == 1:
            # 检查这个唯一的image_id是否也不在文件1的零分集合中
            if image_id not in zero_score_image_ids:
                count += 1
    
    return count

# --- 使用示例 ---
# 请将 'file1.jsonl' 和 'file2.jsonl' 替换为您的实际文件名
file1 = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered.jsonl'
file2 = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_wrong_72b_right.jsonl'


result = count_unique_unseen_images(file1, file2)

if result != -1:
    print(f"在文件 '{file2}' 中，image_id没有在第一个文件中llm_score为0的数据中出现过，且在文件2中只出现一次的数据共有: {result} 条")
    # 预期输出应为 2 (ddd 和 eee)

在文件 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_wrong_72b_right.jsonl' 中，image_id没有在第一个文件中llm_score为0的数据中出现过，且在文件2中只出现一次的数据共有: 1781 条


In [10]:
import json
import random
import os
from collections import Counter

def select_and_split_data(file1_path, file2_path, output_path, num_to_select):
    """
    从file2中筛选符合条件的数据，随机抽取指定数量保存到新文件，并从原文件中删除。

    Args:
        file1_path (str): 第一个jsonl文件的路径。
        file2_path (str): 第二个jsonl文件的路径。
        output_path (str): 用于保存抽取数据的新jsonl文件的路径。
        num_to_select (int): 需要随机抽取的条数。
    
    Returns:
        None
    """
    print("--- 开始处理 ---")

    # --- 步骤 1: 读取文件1，收集所有llm_score为0的image_id ---
    print(f"步骤 1/5: 从 '{os.path.basename(file1_path)}' 中读取零分 image_id...")
    zero_score_image_ids = set()
    try:
        with open(file1_path, 'r', encoding='utf-8') as f1:
            for line in f1:
                try:
                    data = json.loads(line)
                    if data.get("llm_score") == 0:
                        zero_score_image_ids.add(data.get("image_id"))
                except json.JSONDecodeError:
                    continue # 忽略格式错误的行
    except FileNotFoundError:
        print(f"错误：找不到文件 {file1_path}")
        return

    # --- 步骤 2: 预读文件2，统计image_id频率并缓存数据 ---
    print(f"步骤 2/5: 预读并分析 '{os.path.basename(file2_path)}'...")
    image_ids_in_file2 = []
    file2_content = []
    try:
        with open(file2_path, 'r', encoding='utf-8') as f2:
            for line in f2:
                try:
                    data = json.loads(line)
                    image_ids_in_file2.append(data.get("image_id"))
                    file2_content.append(data) # 将解析后的数据缓存到内存
                except json.JSONDecodeError:
                    continue # 忽略格式错误的行
    except FileNotFoundError:
        print(f"错误：找不到文件 {file2_path}")
        return

    image_id_counts = Counter(image_ids_in_file2)

    # --- 步骤 3: 筛选出所有符合条件的数据 ---
    print("步骤 3/5: 筛选所有符合条件的数据...")
    eligible_data = []
    for data in file2_content:
        image_id = data.get("image_id")
        if image_id and image_id_counts.get(image_id) == 1 and image_id not in zero_score_image_ids:
            eligible_data.append(data)
    
    print(f"发现 {len(eligible_data)} 条符合条件的数据。")

    if len(eligible_data) < num_to_select:
        print(f"警告：符合条件的数据（{len(eligible_data)}条）少于需要抽取的数量（{num_to_select}条）。将抽取所有符合条件的数据。")
        num_to_select = len(eligible_data)

    if num_to_select == 0:
        print("没有可供抽取的数据。程序结束。")
        return

    # --- 步骤 4: 随机抽样并写入新文件 ---
    print(f"步骤 4/5: 随机抽取 {num_to_select} 条数据并写入 '{os.path.basename(output_path)}'...")
    selected_data = random.sample(eligible_data, num_to_select)
    selected_image_ids = {item['image_id'] for item in selected_data} # 创建一个集合以便快速查找

    try:
        with open(output_path, 'w', encoding='utf-8') as f_out:
            for item in selected_data:
                f_out.write(json.dumps(item, ensure_ascii=False) + '\n')
    except IOError as e:
        print(f"错误：无法写入到输出文件 {output_path}。原因: {e}")
        return

    # --- 步骤 5: 从文件2中移除已抽样的数据 (通过重写文件实现) ---
    print(f"步骤 5/5: 更新 '{os.path.basename(file2_path)}'，移除已抽样的数据...")
    
    temp_file_path = file2_path + '.tmp'
    try:
        with open(temp_file_path, 'w', encoding='utf-8') as f_temp:
            # 这里我们使用之前缓存的 file2_content 来避免再次读取文件
            for data in file2_content:
                if data.get("image_id") not in selected_image_ids:
                    f_temp.write(json.dumps(data, ensure_ascii=False) + '\n')
        
        # 用新生成的文件安全地替换旧文件
        os.replace(temp_file_path, file2_path)
    except IOError as e:
        print(f"错误：更新文件 {file2_path} 失败。原因: {e}")
        # 如果更新失败，清理临时文件
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)
        return

    print("--- 处理完成 ---")
    print(f"成功抽取 {len(selected_data)} 条数据到 '{output_path}'。")
    print(f"'{file2_path}' 已更新。")


# --- 使用示例 ---
# 请将以下路径替换为您的实际文件路径
file1 = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/72b_responses_llm_filtered.jsonl'
file2 = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_wrong_72b_right.jsonl'

# 定义新文件的保存路径和名称
output_file = '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/validation_962.jsonl'

# 您希望抽取的数量
num_to_extract = 962

# 调用函数
select_and_split_data(file1, file2, output_file, num_to_extract)

--- 开始处理 ---
步骤 1/5: 从 '72b_responses_llm_filtered.jsonl' 中读取零分 image_id...
步骤 2/5: 预读并分析 '7b_wrong_72b_right.jsonl'...
步骤 3/5: 筛选所有符合条件的数据...
发现 1781 条符合条件的数据。
步骤 4/5: 随机抽取 962 条数据并写入 'validation_962.jsonl'...
步骤 5/5: 更新 '7b_wrong_72b_right.jsonl'，移除已抽样的数据...
--- 处理完成 ---
成功抽取 962 条数据到 '/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/validation_962.jsonl'。
'/mnt/petrelfs/sunhaoyu/visual-code/Tool-Data-Curation/web/test_results/guichat/7b_wrong_72b_right.jsonl' 已更新。
